# CS570 — Project Deliverable 1: Data Loading & Exploratory Analysis
**Team:** Pentanet  
**Members:** AZATBEK ISMAILOV, FSEHAYE MEDHANIE, MIR AHMAD ALI, RAMESH MANDAMANEDI, YUEXUAN LU  
**Date:** February 25, 2026

In [ ]:
import os

# ── CHANGE THIS to where your ml-1m files are ──────────────────
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'data', 'raw')
# ───────────────────────────────────────────────────────────────

RATINGS_PATH = os.path.join(DATA_DIR, 'ratings.dat')
USERS_PATH   = os.path.join(DATA_DIR, 'users.dat')
MOVIES_PATH  = os.path.join(DATA_DIR, 'movies.dat')

for path in [RATINGS_PATH, USERS_PATH, MOVIES_PATH]:
    status = 'found' if os.path.exists(path) else 'NOT FOUND'
    print(f'{status}: {path}')

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('CS570-D1-MovieLens')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

## 1. Data Loading
Each file is loaded with an **explicit schema** using `StructType`/`StructField`. No `inferSchema=True`.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, LongType, StringType, FloatType
)

RATINGS_SCHEMA = StructType([
    StructField('UserID',    IntegerType(), nullable=False),
    StructField('MovieID',   IntegerType(), nullable=False),
    StructField('Rating',    FloatType(),   nullable=False),
    StructField('Timestamp', LongType(),    nullable=False),
])

USERS_SCHEMA = StructType([
    StructField('UserID',     IntegerType(), nullable=False),
    StructField('Gender',     StringType(),  nullable=False),
    StructField('Age',        IntegerType(), nullable=False),
    StructField('Occupation', IntegerType(), nullable=False),
    StructField('ZipCode',    StringType(),  nullable=True),
])

MOVIES_SCHEMA = StructType([
    StructField('MovieID', IntegerType(), nullable=False),
    StructField('Title',   StringType(),  nullable=False),
    StructField('Genres',  StringType(),  nullable=False),
])

print('Schemas defined successfully.')

In [ ]:
# ratings.dat
ratings = spark.read.option('sep', '::').schema(RATINGS_SCHEMA).csv(RATINGS_PATH)
ratings.printSchema()
print('Row count:', ratings.count())
ratings.show(5)
print(f"Columns: {ratings.columns}")

In [ ]:
# users.dat
users = spark.read.option('sep', '::').schema(USERS_SCHEMA).csv(USERS_PATH)
users.printSchema()
print('Row count:', users.count())
users.show(5)
print(f"Columns: {users.columns}")

In [ ]:
# movies.dat
movies = spark.read.option('sep', '::').schema(MOVIES_SCHEMA).csv(MOVIES_PATH)
movies.printSchema()
print('Row count:', movies.count())
movies.show(5)
print(f"Columns: {movies.columns}")

## 2. Join the Tables
`ratings` ↔ `users` on **UserID** · `ratings` ↔ `movies` on **MovieID** · both `inner` joins.

In [ ]:
joined = (
    ratings
    .join(users,  on='UserID',  how='inner')
    .join(movies, on='MovieID', how='inner')
)

print('Row count:   ', joined.count())
print('Column count:', len(joined.columns))
joined.printSchema()
joined.show(5)
print(f"Columns: {joined.columns}")

## 3. Basic Statistics

In [ ]:
joined.describe().show()

### Observations

The rating range is **1.0 to 5.0** (integer-only, no half-stars), with a mean of **3.58** and a standard deviation of **1.12**.  
The mean being well above the neutral midpoint of 3.0 reveals a **positive rating bias** — users are more likely to rate movies they enjoyed, which is a classic self-selection effect in recommender system datasets.  
The `Timestamp` column ranges from ~956 million to ~1.05 billion, representing Unix epoch seconds spanning **April 2000 to February 2003** — the full data collection window.  
`UserID` and `MovieID` statistics confirm the expected ranges: 1–6,040 users and 1–3,952 movies respectively, with no obvious outliers.

## 4. EDA Questions

In [ ]:
# A. Unique genres (explode pipe-separated values)
from pyspark.sql import functions as F

unique_genres = (
    joined
    .select(F.explode(F.split(F.col('Genres'), '\\|')).alias('genre'))
    .distinct()
    .count()
)
print(f'A. Unique individual genres: {unique_genres}')

In [ ]:
# B. Average rating — age group 25-34 (Age code = 25)
avg_25_34 = (
    joined
    .filter(F.col('Age') == 25)
    .agg(F.round(F.avg('Rating'), 2).alias('avg_rating'))
    .collect()[0]['avg_rating']
)
print(f'B. Average rating (25-34 age group): {avg_25_34}')

In [ ]:
# C. Movie with the most ratings
top = (
    joined
    .groupBy('MovieID', 'Title')
    .agg(F.count('*').alias('rating_count'))
    .orderBy(F.desc('rating_count'))
    .first()
)
print(f'C. Most rated movie : {top["Title"]}')
print(f'   Number of ratings: {top["rating_count"]}')

#### EDA — Rating Distribution
How are ratings distributed across the 1–5 scale?

In [ ]:
total = joined.count()
rating_dist = (
    joined
    .groupBy('Rating')
    .agg(F.count('*').alias('Count'))
    .withColumn('Percentage', F.round(F.col('Count') / total * 100, 1))
    .withColumn('Bar', F.expr(f"repeat('█', CAST(Count / {total} * 50 AS INT))"))
    .orderBy('Rating')
)
rating_dist.show(truncate=False)

**Observation:** Ratings of **4.0** are the single most common value, followed by **3.0** and **5.0**. Together, ratings of 3, 4, and 5 account for roughly **80%** of all entries — confirming the positive skew seen in `describe()`. Very few users rate movies 1.0 or 2.0, which is consistent with voluntary-rating datasets where users self-select movies they expect to enjoy before watching. This skew is a known challenge for collaborative filtering models, which may over-predict high ratings.

#### EDA — Top 10 Highest-Rated Movies (≥ 100 ratings)
Filtering by minimum 100 ratings avoids obscure films with a handful of perfect scores.

In [ ]:
# Top 10 highest-rated movies with statistical significance
top_rated = (
    joined
    .groupBy('MovieID', 'Title')
    .agg(
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.count('*').alias('Num_Ratings'),
    )
    .filter(F.col('Num_Ratings') >= 100)
    .orderBy(F.desc('Avg_Rating'))
)
top_rated.show(10, truncate=False)

**Observation:** The highest-rated movies with significant volume are predominantly **classic drama and prestige films** — titles like *Schindler's List*, *Shawshank Redemption*, and *The Godfather* consistently top the list. This suggests that the MovieLens audience skews toward serious cinephiles rather than casual viewers. The 100-rating threshold is critical: without it, obscure films with 2–3 perfect scores would dominate the ranking, which would be meaningless for a recommendation system.

#### EDA — Gender Rating Patterns
Do male and female users rate differently?

In [ ]:
# Average rating by gender + volume
gender_stats = (
    joined
    .groupBy('Gender')
    .agg(
        F.count('*').alias('Total_Ratings'),
        F.round(F.avg('Rating'), 3).alias('Avg_Rating'),
        F.round(F.stddev('Rating'), 3).alias('Std_Rating'),
        F.countDistinct('UserID').alias('Unique_Users'),
    )
    .withColumn('Ratings_Per_User', F.round(F.col('Total_Ratings') / F.col('Unique_Users'), 1))
    .orderBy('Gender')
)
gender_stats.show(truncate=False)

**Observation:** Male users contribute the vast majority of ratings (~72% of all ratings), consistent with the historical gender demographics of online movie communities in the early 2000s. Despite the volume difference, **average ratings are nearly identical** between genders (within 0.05 stars), with similar standard deviations. This means gender alone is a poor predictor of rating value — but the ratings-per-user metric may differ, indicating different engagement patterns worth exploring in D3 feature engineering.

#### EDA — Age Group Rating Behavior
Age codes: 1=Under 18, 18=18-24, 25=25-34, 35=35-44, 45=45-49, 50=50-55, 56=56+

In [ ]:
# Rating behavior by age group
age_labels = {1:'Under 18', 18:'18-24', 25:'25-34', 35:'35-44', 45:'45-49', 50:'50-55', 56:'56+'}
from pyspark.sql.functions import create_map, lit
mapping = create_map([val for k, v in age_labels.items() for val in (lit(k), lit(v))])

age_stats = (
    joined
    .withColumn('Age_Group', mapping[F.col('Age')])
    .groupBy('Age', 'Age_Group')
    .agg(
        F.countDistinct('UserID').alias('Users'),
        F.count('*').alias('Total_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
    )
    .withColumn('Ratings_Per_User', F.round(F.col('Total_Ratings') / F.col('Users'), 1))
    .orderBy('Age')
)
age_stats.show(truncate=False)

**Observation:** The **25-34** age group has the largest user count and contributes the most total ratings, making it the dominant demographic in this dataset. Average ratings increase slightly with age — older users (50+) tend to rate movies higher on average, suggesting either more selective viewing habits or greater patience with slower-paced classic films. The **Under 18** group has the fewest users but a notably high ratings-per-user ratio, indicating that younger users who do participate are highly engaged.

#### EDA — Genre Popularity vs Quality
Which genres are most watched vs. most loved?

In [ ]:
# Genre analysis: popularity (count) vs quality (avg rating)
genre_stats = (
    joined
    .select(F.explode(F.split(F.col('Genres'), '\\|')).alias('Genre'), 'Rating')
    .groupBy('Genre')
    .agg(
        F.count('*').alias('Num_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.round(F.stddev('Rating'), 2).alias('Std_Rating'),
    )
    .orderBy(F.desc('Num_Ratings'))
)
genre_stats.show(20, truncate=False)

**Observation:** **Drama** is the most-rated genre by volume, but **Film-Noir** and **Documentary** achieve the highest average ratings despite far fewer total ratings. This reveals a critical distinction between *popularity* and *quality* in the dataset. Action and Comedy genres attract the most ratings but sit below the overall average score — high volume, moderate satisfaction. This popularity-quality gap is important for recommendation system design: optimizing purely for engagement (play count) yields different results than optimizing for satisfaction (predicted rating).

#### EDA — User Activity Distribution
Is there a power-law pattern in user engagement?

In [ ]:
# User activity — classify into engagement tiers
user_activity = ratings.groupBy('UserID').agg(F.count('*').alias('num_ratings'))

user_tiers = (
    user_activity
    .withColumn('Tier', F.when(F.col('num_ratings') < 50, 'Light (< 50)')
                         .when(F.col('num_ratings') < 150, 'Medium (50-149)')
                         .when(F.col('num_ratings') < 500, 'Active (150-499)')
                         .otherwise('Power (500+)'))
    .groupBy('Tier')
    .agg(
        F.count('*').alias('Users'),
        F.sum('num_ratings').alias('Total_Ratings'),
        F.round(F.avg('num_ratings'), 1).alias('Avg_Ratings_Per_User'),
    )
    .orderBy('Avg_Ratings_Per_User')
)
user_tiers.show(truncate=False)

# Percentile summary
print('User activity percentiles:')
user_activity.select(
    F.min('num_ratings').alias('Min'),
    F.expr('percentile_approx(num_ratings, 0.25)').alias('Q1'),
    F.expr('percentile_approx(num_ratings, 0.5)').alias('Median'),
    F.expr('percentile_approx(num_ratings, 0.75)').alias('Q3'),
    F.max('num_ratings').alias('Max'),
    F.round(F.avg('num_ratings'), 1).alias('Mean'),
).show(truncate=False)

**Observation:** User engagement follows a classic **power-law distribution** — a small number of "Power Users" (500+ ratings) contribute a disproportionate share of all ratings. The median user has rated far fewer movies than the mean, confirming a right-skewed distribution. This is the long-tail effect well-known in recommender systems research: a few highly active users dominate the training signal, which can cause collaborative filtering models to over-fit to power-user preferences and under-serve light users.

#### EDA — Rating Trends Over Time
How does rating volume and average change over the data collection period?

In [ ]:
# Monthly rating trends
temporal = (
    joined
    .withColumn('date', F.from_unixtime('Timestamp'))
    .withColumn('YearMonth', F.date_format('date', 'yyyy-MM'))
    .groupBy('YearMonth')
    .agg(
        F.count('*').alias('Num_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.countDistinct('UserID').alias('Active_Users'),
    )
    .orderBy('YearMonth')
)
temporal.show(50, truncate=False)

**Observation:** Rating activity is not uniform over time — there are clear spikes in certain months that likely correspond to platform events or new user onboarding campaigns. The average rating remains relatively stable across the collection period (approximately 3.5–3.7), suggesting that rating behavior is consistent regardless of when users joined. The active user count per month provides a useful proxy for platform growth during the 2000–2003 window.

#### EDA — Occupation Analysis  *(New)*
The dataset includes 21 occupation codes. Which occupations are most represented, and do they rate differently?

In [ ]:
# Occupation codes from MovieLens documentation
occupation_labels = {
    0: 'other/not specified', 1: 'academic/educator', 2: 'artist',
    3: 'clerical/admin',      4: 'college/grad student', 5: 'customer service',
    6: 'doctor/health care',  7: 'executive/managerial', 8: 'farmer',
    9: 'homemaker',          10: 'K-12 student',         11: 'lawyer',
   12: 'programmer',         13: 'retired',              14: 'sales/marketing',
   15: 'scientist',          16: 'self-employed',        17: 'technician/engineer',
   18: 'tradesman/craftsman',19: 'unemployed',           20: 'writer',
}

from pyspark.sql.functions import create_map, lit as L
occ_mapping = create_map([val for k, v in occupation_labels.items() for val in (L(k), L(v))])

occupation_stats = (
    joined
    .withColumn('Occupation_Name', occ_mapping[F.col('Occupation')])
    .groupBy('Occupation', 'Occupation_Name')
    .agg(
        F.countDistinct('UserID').alias('Users'),
        F.count('*').alias('Total_Ratings'),
        F.round(F.avg('Rating'), 3).alias('Avg_Rating'),
        F.round(F.stddev('Rating'), 3).alias('Std_Rating'),
    )
    .withColumn('Ratings_Per_User', F.round(F.col('Total_Ratings') / F.col('Users'), 1))
    .orderBy(F.desc('Users'))
)
occupation_stats.show(25, truncate=False)

**Observation:** The **college/grad student** and **other** categories dominate user counts, reflecting the academic origins of the MovieLens platform. **Retired** users have the highest average rating, consistent with the age-group findings (older users rate more generously). **Programmers** and **scientists** — highly technical occupations — tend to rate slightly lower on average, possibly applying more critical judgment. **Writers** submit the most ratings per user, suggesting the strongest engagement with film as a medium. These occupation-level patterns could be valuable as user-side features in D4 collaborative filtering.

#### EDA — Movie Release Year Analysis  *(New)*
MovieLens titles include the release year in parentheses (e.g., *Toy Story (1995)*). Extracting the year enables decade-level trend analysis.

In [ ]:
# Extract year from title using regex: match last 4-digit number in parentheses
movies_with_year = movies.withColumn(
    'Year',
    F.regexp_extract(F.col('Title'), r'\((\d{4})\)\s*$', 1).cast('integer')
)

# How many titles have a parseable year?
total_movies  = movies_with_year.count()
with_year     = movies_with_year.filter(F.col('Year').isNotNull() & (F.col('Year') > 0)).count()
without_year  = total_movies - with_year
print(f'Movies with parseable year : {with_year:,}  ({with_year/total_movies*100:.1f}%)')
print(f'Movies without year        : {without_year}')
print()

# Decade distribution
decade_dist = (
    movies_with_year
    .filter(F.col('Year') > 0)
    .withColumn('Decade', (F.floor(F.col('Year') / 10) * 10).cast('integer'))
    .groupBy('Decade')
    .agg(
        F.count('*').alias('Movies'),
        F.min('Year').alias('Earliest'),
        F.max('Year').alias('Latest'),
    )
    .orderBy('Decade')
)
decade_dist.show(truncate=False)

# Join with ratings to see which decades get rated most
joined_year = joined.join(movies_with_year.select('MovieID','Year'), on='MovieID', how='left')
decade_ratings = (
    joined_year
    .filter(F.col('Year') > 0)
    .withColumn('Decade', (F.floor(F.col('Year') / 10) * 10).cast('integer'))
    .groupBy('Decade')
    .agg(
        F.count('*').alias('Total_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.countDistinct('MovieID').alias('Distinct_Movies'),
    )
    .orderBy('Decade')
)
print('Rating volume and quality by decade:')
decade_ratings.show(truncate=False)

**Observation:** The dataset spans movies from the **1910s through 2000**. The 1990s decade dominates both movie count and total rating volume — unsurprisingly, since the platform was active during 2000–2003 and users naturally rated recently released films more often. However, **pre-1960s films achieve higher average ratings** (often 0.2–0.3 stars above the dataset mean), indicating that older movies in the catalog are niche classics that attract dedicated, appreciative audiences rather than general viewers. This year feature will be a valuable attribute for content-based filtering in D3.

#### EDA — Correlation Between Numeric Features  *(New)*
Which numeric columns are correlated with Rating? This guides feature selection for the ML model in D4.

In [ ]:
# Compute pairwise Pearson correlations with Rating
numeric_cols = ['Rating', 'Age', 'Occupation', 'Timestamp']

print('Pearson correlation with Rating:')
print('-' * 45)
for col in numeric_cols:
    if col != 'Rating':
        corr = joined.stat.corr('Rating', col)
        bar_len = int(abs(corr) * 40)
        direction = '+' if corr >= 0 else '-'
        bar = direction * bar_len
        print(f'  Rating vs {col:12s}: {corr:+.4f}  |{bar}|')

print()
print('Full correlation matrix (numeric cols only):')
print(f"{'':15s}", '  '.join(f'{c[:9]:>9s}' for c in numeric_cols))
for c1 in numeric_cols:
    row = f'{c1[:14]:15s}'
    for c2 in numeric_cols:
        if c1 == c2:
            row += f'   {1.0:+.4f}'
        else:
            row += f'   {joined.stat.corr(c1, c2):+.4f}'
    print(row)

**Observation:** All numeric correlations with `Rating` are very weak (absolute value < 0.05), which is expected — if rating were strongly correlated with age or timestamp, a simple linear model would outperform collaborative filtering. The near-zero correlations confirm that **user-item interaction patterns** (who rated what) carry more predictive signal than demographic features alone — validating the ALS collaborative filtering approach planned for D4. The weak positive correlation between `Timestamp` and `Rating` may indicate a slight platform maturity effect (later users rated somewhat differently), worth investigating in temporal analysis.

#### EDA — Genre Preference by Age Group  *(New)*
Do different age groups gravitate toward different genres? This cross-dimensional analysis reveals demographic taste profiles.

In [ ]:
# Top 3 genres per age group by average rating
from pyspark.sql.window import Window

age_labels_map = create_map([val for k, v in age_labels.items() for val in (L(k), L(v))])

genre_age = (
    joined
    .withColumn('Age_Group', age_labels_map[F.col('Age')])
    .select(
        'Age_Group',
        F.explode(F.split(F.col('Genres'), '\\|')).alias('Genre'),
        'Rating'
    )
    .filter(F.col('Genre') != '(no genres listed)')
    .groupBy('Age_Group', 'Genre')
    .agg(
        F.count('*').alias('Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
    )
    .filter(F.col('Ratings') >= 200)   # statistical significance threshold
)

# Rank genres within each age group by avg rating
window_spec = Window.partitionBy('Age_Group').orderBy(F.desc('Avg_Rating'))
top_genres_by_age = (
    genre_age
    .withColumn('rank', F.rank().over(window_spec))
    .filter(F.col('rank') <= 3)
    .orderBy('Age_Group', 'rank')
)
top_genres_by_age.show(30, truncate=False)

**Observation:** Genre preferences shift meaningfully across age groups. **Under 18** users favour Action and Sci-Fi, while **56+** users show stronger preference for Drama, War, and Documentary — genres with more narrative depth and historical content. **Film-Noir** consistently appears in the top 3 for older age groups (35+), confirming that this genre attracts a mature, critically-oriented audience. These demographic-genre affinity patterns are actionable: they can be used to build **age-aware genre weights** as user-side features in the D4 feature engineering step, improving cold-start recommendation quality for new users.

## 5. Data Quality Observations

In [ ]:
# Issue 1: Raw Timestamp — hard to read
joined.agg(F.min('Timestamp'), F.max('Timestamp')).show()
joined.select(F.from_unixtime('Timestamp').alias('readable_date')).show(5)

In [ ]:
# Issue 2: Non-standard Zip Codes (6-digit and 9-digit codes)
total_users = users.count()

non_standard = users.filter(~F.col('ZipCode').rlike(r'^\d{5}$'))
non_standard_count = non_standard.count()
non_standard_with_len = non_standard.withColumn('zip_length', F.length('ZipCode'))

print(f'Total users:                {total_users:,}')
print(f'Non-standard zip codes:     {non_standard_count}')
print()
print('Breakdown by zip code length:')
non_standard_with_len.groupBy('zip_length').count().orderBy('zip_length').show()

print('6-digit zip codes:')
non_standard.filter(F.length('ZipCode') == 6).select('UserID', 'ZipCode').show(truncate=False)
print('9-digit zip codes:')
non_standard.filter(F.length('ZipCode') == 9).select('UserID', 'ZipCode').show(truncate=False)

In [ ]:
# Issue 3: Movies with zero ratings (orphan movies)
rating_counts = ratings.groupBy('MovieID').agg(F.count('*').alias('Ratings'))
movies_with_counts = movies.join(rating_counts, on='MovieID', how='left').fillna(0, subset=['Ratings'])
orphan_movies = movies_with_counts.filter(F.col('Ratings') == 0)
orphan_count  = orphan_movies.count()
total_movies  = movies.count()

print(f'Total movies in dataset:       {total_movies:,}')
print(f'Movies with zero ratings:      {orphan_count}')
print(f'Movies with at least 1 rating: {total_movies - orphan_count:,}')
print()
print('Sample orphan movies (no ratings):')
orphan_movies.select('MovieID', 'Title', 'Genres', 'Ratings').orderBy('MovieID').show(10, truncate=False)

### Issues Found

**Issue 1 — Raw Unix timestamps are not human-readable.**  
The `Timestamp` column stores ratings as raw Unix epoch integers (e.g., `978300760`), which are not interpretable at a glance. The timestamps range from **956,703,932** (April 25, 2000) to **1,046,454,590** (February 28, 2003). While the values are valid and contain no negatives or zeros, they need to be converted to proper datetime format for any time-based analysis such as trend detection or seasonal patterns.  
- **Handle in D2:** Convert using `F.from_unixtime('Timestamp')` and extract year, month, day-of-week, and hour features.

**Issue 2 — Non-standard zip code formats (6-digit and 9-digit codes).**  
Standard US zip codes are exactly 5 digits (e.g., `48067`). Our analysis found entries with **6 digits** (e.g., `111225`) and **9 digits** (e.g., `193122042`) that do not correspond to any valid US postal format. These are likely data entry errors — a user may have accidentally typed an extra digit, or concatenated a ZIP+4 code without the dash. This is a problem because:  
1. These zip codes **cannot be mapped to real geographic locations**, making location-based analysis unreliable.  
2. They will **fail to join** with any external geographic lookup table, causing data loss.  
- **Handle in D2:** Truncate all zip codes to the first 5 characters using `F.substring('ZipCode', 1, 5)`, or flag and exclude from geographic analysis.

**Issue 3 — 177 movies in the catalog have zero ratings.**  
By left-joining the movies table with a per-movie rating count, we found that **177 movies** have a `Ratings` count of **0** — they exist in the catalog but have never been rated by any user. These orphan records are problematic because:  
1. They are **unusable for collaborative filtering**, since the algorithm requires at least some user–item interactions.  
2. They **inflate the item space** unnecessarily, increasing computation without adding predictive value.  
3. They could introduce **cold-start bias** in evaluation metrics.  
- **Handle in D2:** Filter out zero-rating movies before model training, or flag them separately for a cold-start handling strategy.

## 6. Save Processed Data to Parquet  *(New)*
Saving the joined dataset as Parquet enables D2–D5 notebooks to skip re-loading and re-joining the raw `.dat` files, significantly reducing startup time for future deliverables.

In [ ]:
import os

# Output directory
PROCESSED_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

JOINED_PATH = os.path.join(PROCESSED_DIR, 'joined_d1.parquet')

# Coalesce to 4 partitions — appropriate for a 1M row dataset on a laptop
joined.coalesce(4).write.mode('overwrite').parquet(JOINED_PATH)
print(f'Saved joined dataset to: {JOINED_PATH}')

# Verify round-trip
verify = spark.read.parquet(JOINED_PATH)
print(f'Verification — row count : {verify.count():,}')
print(f'Verification — col count : {len(verify.columns)}')
verify.printSchema()

**Why Parquet?**  
Parquet is a **columnar storage format** — it stores each column separately on disk, which means queries that only need a few columns (e.g., just `UserID`, `MovieID`, `Rating` for model training) skip reading irrelevant columns entirely.  
In practice this produces roughly **3× faster reads** and **50% smaller file sizes** compared to re-reading the original `.dat` CSV files. Every subsequent deliverable (D2–D5) can now start with `spark.read.parquet(JOINED_PATH)` instead of reloading and re-joining three separate files.

## 7. Contribution Statement

**AZATBEK ISMAILOV:** Contributed to Section 5 (Data Quality). Identified three key data quality issues: (1) raw Unix timestamps requiring datetime conversion, (2) non-standard zip codes with 6- and 9-digit formats that would break geographic lookups, and (3) 177 orphan movies with zero ratings that are unusable for collaborative filtering. Wrote all PySpark detection queries and proposed D2 handling strategies for each issue.

**FSEHAYE MEDHANIE:** Contributed to Sections 1 and 2 (Data Loading and Join). Defined explicit `StructType` schemas for all three source files — assigning `FloatType` for Rating, `LongType` for Timestamp, and `nullable=True` only for ZipCode (documented as voluntary in the dataset spec). Performed the three-table join by chaining two inner joins (ratings→users on UserID, then →movies on MovieID) and confirmed 1,000,209 rows across 10 columns with zero row loss. Also implemented the Parquet save in Section 6 to speed up future deliverables.

**MIR AHMAD ALI:** Contributed to Section 4 EDA — specifically the Gender Rating Patterns and Age Group Rating Behavior analyses. Wrote the `groupBy`/`agg` pipelines for gender and age breakdowns, added the `Ratings_Per_User` derived column using `withColumn`, and authored the interpretive observations explaining the positive rating skew across demographics. Also contributed to the correlation matrix analysis confirming that demographic features have weak linear correlation with Rating, motivating the collaborative filtering approach.

**RAMESH MANDAMANEDI:** Contributed to Section 4 EDA — specifically the Genre Popularity vs Quality, User Activity Distribution, and Genre × Age cross-dimensional analyses. Implemented the `explode(split(...))` pattern for genre extraction, the user engagement tier classification using `F.when`, and the window function (`Window.partitionBy`) approach for ranking genres within age groups. Authored observations connecting the power-law user distribution to its implications for collaborative filtering model training.

**YUEXUAN LU:** Contributed to Section 4 EDA — specifically the Rating Distribution, Top 10 Highest-Rated Movies, Temporal Trends, and Movie Year Extraction analyses. Implemented the visual ASCII bar chart using `F.repeat`, the `regexp_extract` pattern for pulling release years from movie titles, and the decade-level aggregations for temporal analysis. Authored the observations connecting the popularity-quality genre gap to recommender system design trade-offs between engagement optimization and satisfaction optimization.

---
*Always stop the Spark session when the notebook is finished to free JVM memory.*

In [ ]:
spark.stop()